## **source cell :**

In [2]:
import numpy as np
import random as rd
import pandas as pd
import math
import secrets as st
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import itertools
from scipy.spatial import cKDTree
import random

#### simulation :

In [24]:
time_step = 1
v = 0.3
T = 100
def initialize() :
    global N, L, positions, directions, order_parameters
    positions = np.zeros((N, T, 2))
    directions = np.zeros((N, T))
    positions[:, 0, :] = np.random.uniform(0, L, (N, 2))
    directions[:, 0] = np.random.uniform(0, 2*np.pi, (N, ))

def update(t) :
    global R, v, etta, new_xs, new_ys, new_dirs, order_parameter

    #positions update
    v_vector = np.column_stack([
        np.cos(directions[:, t-1]),
        np.sin(directions[:, t-1])
    ])
    positions[:, t, :] = (positions[:, t-1, :] + v_vector*time_step) % L

    #neighbors
    tree = cKDTree(positions[:, t-1, :], boxsize = L)
    neighbors_lists = tree.query_ball_tree(tree, R)

    #direction update
    for i in range(len(neighbors_lists)) :
        x_dirs = np.cos(directions[neighbors_lists[i], t-1])
        y_dirs = np.sin(directions[neighbors_lists[i], t-1])
        directions[i, t] = (np.arctan2(np.mean(y_dirs), np.mean(x_dirs))
                            + random.uniform(-etta/2, etta/2))

In [30]:
def run_simulation() :
  global directions
  initialize()
  for t in range(1, 100) :
    update(t)

#### parameters :

In [34]:
L = 5
v = 0.3
Rs = np.array([0.5, 1, 1.5, 2, 2.5])
ettas = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5])
densities = np.linspace(1, 40, 250)

In [35]:
R_etta_density_combinations = list(itertools.product(Rs, ettas, densities))
print(len(R_etta_density_combinations))
print(R_etta_density_combinations[0])

10000
(np.float64(0.5), np.float64(0.1), np.float64(1.0))


#### data producing

In [33]:
count = len(R_etta_density_combinations)
dataset = []
for sample in range(len(R_etta_density_combinations)) :
  global L, v
  R = R_etta_density_combinations[sample][0]
  etta = R_etta_density_combinations[sample][1]
  density = R_etta_density_combinations[sample][2]
  N = int(density*L**2)
  run_simulation()
  individual_particle = rd.randint(0, N-1)
  start_time = random.randint(0, 80)
  individuals_dirs = directions[individual_particle, start_time : start_time+20]
  data = [individuals_dirs, density]
  dataset.append(data)
  print(count)
  count-=1

100000
99999
99998
99997
99996
99995
99994
99993
99992
99991
99990
99989
99988
99987
99986
99985
99984
99983
99982
99981
99980
99979
99978
99977
99976
99975
99974
99973
99972
99971
99970
99969
99968
99967
99966
99965
99964
99963
99962
99961
99960
99959
99958
99957
99956
99955
99954
99953
99952
99951
99950
99949
99948
99947
99946
99945
99944
99943
99942
99941
99940
99939
99938
99937
99936
99935
99934
99933
99932
99931
99930
99929
99928
99927
99926
99925
99924
99923
99922
99921
99920
99919
99918
99917
99916
99915
99914
99913
99912
99911
99910
99909
99908
99907
99906
99905
99904
99903
99902
99901
99900
99899
99898
99897
99896
99895
99894
99893
99892
99891
99890
99889
99888
99887
99886
99885
99884
99883
99882
99881
99880
99879
99878
99877
99876
99875
99874
99873
99872
99871
99870
99869
99868
99867
99866
99865
99864
99863
99862
99861
99860
99859
99858
99857
99856
99855
99854
99853
99852
99851
99850
99849
99848
99847
99846
99845
99844
99843
99842
99841
99840
99839
99838
99837
99836
99835
998

KeyboardInterrupt: 

In [29]:
df = pd.DataFrame({
    'individuals_dirs' : dataset[0],
    'density' : dataset[1],
})
df.to_parquet('dataset_density.parquet')

IndexError: list index out of range